In [10]:
import pandas as pd

# =========================
# 1. LOAD RAW CSV FILES
# =========================
happy = pd.read_csv("/Users/ale/Desktop/MySQL/Project-3/world_happiness_report.csv")
econ = pd.read_csv("/Users/ale/Desktop/MySQL/Project-3/Economic Indicators And Inflation.csv")

# =========================
# 2. CLEAN COLUMN NAMES
# =========================
happy.columns = happy.columns.str.strip().str.lower().str.replace(" ", "_")
econ.columns = econ.columns.str.strip().str.lower().str.replace(" ", "_")

# =========================
# 3. STANDARDIZE COUNTRY NAMES
# =========================
country_map = {
    "USA": "United States",
    "US": "United States",
    "UK": "United Kingdom"
}

happy["country"] = happy["country"].replace(country_map).str.strip()
econ["country"] = econ["country"].replace(country_map).str.strip()

# =========================
# 4. FIX YEAR DATA TYPE
# =========================
happy["year"] = pd.to_numeric(happy["year"], errors="coerce")
econ["year"] = pd.to_numeric(econ["year"], errors="coerce")

happy = happy.dropna(subset=["year", "country"])
econ = econ.dropna(subset=["year", "country"])

happy["year"] = happy["year"].astype(int)
econ["year"] = econ["year"].astype(int)

# =========================
# 5. REMOVE DUPLICATES
# =========================
happy = happy.drop_duplicates()
econ = econ.drop_duplicates()

# =========================
# 6. CONVERT NUMERIC COLUMNS
# =========================
for col in happy.columns:
    if col not in ["country", "year"]:
        happy[col] = pd.to_numeric(happy[col], errors="coerce")

for col in econ.columns:
    if col not in ["country", "year"]:
        econ[col] = pd.to_numeric(econ[col], errors="coerce")

# =========================
# 7. CREATE DIMENSION TABLES
# =========================
countries = pd.DataFrame({
    "country_name": sorted(set(happy["country"]).union(set(econ["country"])))
})
countries.insert(0, "country_id", range(1, len(countries) + 1))

years = pd.DataFrame({
    "year": sorted(set(happy["year"]).union(set(econ["year"])))
})

# =========================
# 8. MERGE IDs INTO RAW DATA
# =========================
happy = happy.merge(countries, left_on="country", right_on="country_name", how="left")
econ = econ.merge(countries, left_on="country", right_on="country_name", how="left")

# =========================
# 9. CREATE HAPPINESS TABLE
# =========================
happiness_metrics = happy[[
    "country_id",
    "year",
    "happiness_score",
    "life_satisfaction",
    "social_support",
    "healthy_life_expectancy",
    "freedom",
    "generosity",
    "corruption_perception",
    "public_trust",
    "mental_health_index",
    "work_life_balance",
    "climate_index"
]].copy()

happiness_metrics.insert(0, "happiness_id", range(1, len(happiness_metrics) + 1))

# =========================
# 10. CREATE SOCIAL/ECONOMIC TABLE
# =========================
social_economic_metrics = happy[[
    "country_id",
    "year",
    "gdp_per_capita",
    "population",
    "urbanization_rate",
    "education_index",
    "unemployment_rate",
    "employment_rate",
    "income_inequality",
    "public_health_expenditure",
    "internet_access",
    "crime_rate",
    "political_stability"
]].copy()

social_economic_metrics.insert(0, "social_id", range(1, len(social_economic_metrics) + 1))

# =========================
# 11. CREATE MACROECONOMIC TABLE
# =========================
macroeconomic_indicators = econ[[
    "country_id",
    "year",
    "gdp_(in_billion_usd)",
    "inflation_rate_(%)",
    "unemployment_rate_(%)",
    "economic_growth_(%)"
]].copy()

macroeconomic_indicators.columns = [
    "country_id",
    "year",
    "gdp_billion_usd",
    "inflation_rate_pct",
    "unemployment_rate_pct",
    "economic_growth_pct"
]

macroeconomic_indicators.insert(0, "macro_id", range(1, len(macroeconomic_indicators) + 1))

# =========================
# 12. HANDLE NULLS
# =========================
for df in [happiness_metrics, social_economic_metrics, macroeconomic_indicators]:
    numeric_cols = df.select_dtypes(include="number").columns
    for col in numeric_cols:
        if col not in ["happiness_id", "social_id", "macro_id", "country_id", "year"]:
            df[col] = df[col].fillna(df[col].median())

# =========================
# 13. FORCE ALL FINAL COLUMN NAMES TO LOWERCASE
# =========================
for df in [countries, years, happiness_metrics, social_economic_metrics, macroeconomic_indicators]:
    df.columns = df.columns.str.lower()

# =========================
# 14. EXPORT ALL CSV FILES
# =========================
output_folder = "/Users/ale/Desktop/MySQL/Project-3/"

countries.to_csv(output_folder + "countries.csv", index=False)
years.to_csv(output_folder + "years.csv", index=False)
happiness_metrics.to_csv(output_folder + "happiness_metrics.csv", index=False)
social_economic_metrics.to_csv(output_folder + "social_economic_metrics.csv", index=False)
macroeconomic_indicators.to_csv(output_folder + "macroeconomic_indicators.csv", index=False)

print("All 5 CSV files created successfully.")

All 5 CSV files created successfully.
